# Coffee17 Representation Screening V1

Five-fold matched development screening for MFR, LRBP, adaptive multistage fusion, fusion loss, DWT dual-domain, FFC and global-filter candidates. Outer test is not accessed.


In [ ]:
NOTEBOOK_BUILD = "REPRESENTATION-SCREEN-V1"
SCIENTIFIC_CODE_COMMIT = "35a85897bd6c2e43f1fb097bb80120315edb60e9"

import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-representation-screen-v1"
PROJECT = WORK / "coffee17-representation-screening-v1"
OUT = PROJECT / "experiments"
CONFIG_REL = "configs/representation_screening/REPRESENTATION_SCREENING_V1.yaml"
CANDIDATES = ["R1", "R2", "R3", "R4", "R5", "W7", "F1", "F2"]

assert INPUT.is_dir() and WORK.is_dir(), "Notebook harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command, cwd=cwd, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)

if REPO.exists():
    shutil.rmtree(REPO)
run(["git", "clone", "--quiet", "https://github.com/ediprin/coffee-bean-classification.git", REPO])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])
run([sys.executable, "-m", "pip", "install", "-q", "-r", REPO / "requirements/preprocessing-study.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU.")
print("GPU:", torch.cuda.get_device_name(0))

run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_frequency_screening.py",
    "tests/test_representation_screening.py",
], cwd=REPO)

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import materialize_preprocessing_development

PROJECT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        f"Input harus Coffee17 original 979/17; observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_representation_screen_original.zip"
PROV = WORK / "coffee17_representation_screen_provenance"
CANONICAL = WORK / "coffee17_representation_screen_original_v1"
FOLDS = WORK / "coffee17_representation_screen_folds"

for path in (PROV, CANONICAL, FOLDS):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(ARCHIVE, PROV, canonical_root=CANONICAL)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV / "coffee17_provenance.json",
    FOLDS,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

CONFIG = REPO / CONFIG_REL
for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_representation_screen_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)
    materialize_preprocessing_development(
        CANONICAL,
        FOLDS / "clean_manifest.json",
        FOLDS / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_representation_screening_v1",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--candidates", *CANDIDATES,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--strict-determinism",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"fold_{fold}.log")

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

SUMMARY = PROJECT / "analysis" / "representation_screening_v1.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_representation_screening_summary_v1",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
print("\n=== AGGREGATE ===")
for candidate in summary["candidates"]:
    item = summary["aggregate"][candidate]
    macro = item["metrics"]["macro_f1"]["mean"]
    delta = item["delta_vs_anchor"]["macro_f1"]["mean"]
    pos = item["positive_delta_folds"]["macro_f1"]
    hard = item["delta_vs_anchor"]["hard_class_f1"]["mean"]
    worst = item["delta_vs_anchor"]["worst_class_f1"]["mean"]
    print(
        f"{candidate:>2} vs {item['anchor']}: "
        f"Macro={macro:.4%} dMacro={delta:+.4%} pos={pos}/5 "
        f"dHard={hard:+.4%} dWorst={worst:+.4%} "
        f"params={item['parameter_count']:,}"
    )

zip_path = Path(shutil.make_archive(
    str(WORK / "representation-screening-v1"),
    "zip",
    root_dir=PROJECT,
))
print("\nREADY:", zip_path)
print("SIZE:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("Outer test was NOT accessed.")
